In [ ]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 10.0 MB/s eta 0:00:00


In [ ]:
# Upload reza_corpus.txt to Colab first (Files panel on the left)
# Or paste the path if using Drive

with open('reza_corpus.txt', 'r') as f:
    corpus = f.read()

print(f"Corpus loaded: {len(corpus)} chars")
print(corpus[:500])

Corpus loaded: 43211 chars
[WHATSAPP]

by CMU do you mean Carnegie Mellon?

---

Just as I started going back to uni it got cancelled

---

We got some good snow here! But I have two back to back exams on Saturday and Sunday so couldn't really enjoy as much

---

Ok ok keep me updated

---

Any specific time or event you want to go here?

---

I'll send you live location when we get there

---

Going to Creed III can u join?

---

Yo you watched secret invasion?

---

Hala what was the answer?😁 do you assume their benzes 


In [ ]:
SYSTEM_PROMPT = """You are an expert computational psycholinguist and writing style analyst.

Your job is to deeply analyze a corpus of someone's real writing and extract a structured psycholinguistic style profile. You separate STYLE from CONTENT — you are not interested in what they write about, only HOW they write.

You must return ONLY valid JSON. No preamble, no explanation, no markdown code fences. Just the raw JSON object."""

EXTRACTION_PROMPT = """Analyze the following writing corpus and extract a detailed psycholinguistic style profile.

The corpus is labeled by source: [WHATSAPP] for casual messages, [SLACK] for professional-casual messages, [EMAIL] for formal and semi-formal emails. Messages are separated by ---.

Return ONLY this JSON structure with all fields filled in based on evidence from the corpus. Be specific and evidence-based — use actual phrases and patterns you observe, not generalizations.

For fields marked with pattern+examples structure, always provide:
- "examples": actual instances observed in the corpus
- "pattern": the underlying habit those examples reflect, described abstractly so it can be reproduced without copying

{{
  "global": {{
    "formality_score": <float 0.0-1.0 where 0=very casual, 1=very formal>,
    "register": "<descriptor>",
    "avg_message_length": "<very short|short|medium|long>",
    "length_variance": "<low|medium|high>",
    "primary_contexts": ["<context1>", "<context2>"],
    "language_mixing": <true|false>,
    "overall_tone": "<descriptor>",
    "social_orientation": "<self-focused|other-focused|balanced>",
    "politeness_strategy": "<positive politeness|negative politeness|mixed>"
  }},
  "mid_level": {{
    "sentence_rhythm": "<fragmented|flowing|mixed>",
    "avg_sentences_per_message": <float>,
    "uses_bullet_points": <true|false>,
    "uses_numbered_lists": <true|false>,
    "paragraph_structure": "<descriptor>",
    "question_frequency": "<low|medium|high|very high>",
    "rhetorical_questions": <true|false>,
    "opener_patterns": {{
      "examples": ["<actual opener 1>", "<actual opener 2>", "<actual opener 3>"],
      "pattern": "<abstract description of the opening habit>"
    }},
    "closer_patterns": {{
      "examples": ["<actual closer 1>", "<actual closer 2>"],
      "pattern": "<abstract description of the closing habit>"
    }},
    "structural_habits": ["<observed habit 1>", "<observed habit 2>", "<observed habit 3>"],
    "information_structure": "<descriptor of how they order content>",
    "context_switching": "<smooth|abrupt|signaled>",
    "follow_up_behavior": "<descriptor>",
    "social_maintenance_frequency": "<low|medium|high>",
    "topic_management": "<descriptor>"
  }},
  "local": {{
    "emoji_usage": "<none|rare|moderate|frequent>",
    "emoji_style": {{
      "examples": ["<actual emoji with context>"],
      "pattern": "<descriptor of when and why they use emoji>"
    }},
    "exclamation_frequency": "<low|medium|high|very high>",
    "question_mark_style": "<descriptor>",
    "period_usage": "<low|medium|high>",
    "comma_usage": "<descriptor>",
    "capitalization": "<descriptor>",
    "typo_tolerance": "<low|medium|high>",
    "typo_patterns": ["<observed pattern 1>", "<observed pattern 2>"],
    "prosodic_compensation": {{
      "letter_repetition": {{
        "examples": ["<actual examples from corpus>"],
        "pattern": "<when and why they elongate letters>"
      }},
      "punctuation_stacking": {{
        "examples": ["<actual examples from corpus>"],
        "pattern": "<when and why they stack punctuation>"
      }},
      "caps_for_emphasis": <true|false>
    }},
    "filler_phrases": {{
      "examples": ["<actual phrase 1>", "<actual phrase 2>"],
      "pattern": "<abstract description of the filler habit>"
    }},
    "hedging_language": {{
      "examples": ["<actual phrase 1>", "<actual phrase 2>"],
      "pattern": "<descriptor of when and how they hedge>"
    }},
    "intensifiers": {{
      "examples": ["<actual word 1>", "<actual word 2>"],
      "pattern": "<descriptor of intensifier usage>"
    }},
    "pronoun_ratio": {{
      "I_frequency": "<low|medium|high>",
      "you_frequency": "<low|medium|high>",
      "we_frequency": "<low|medium|high>"
    }},
    "vocabulary_level": "<basic|accessible|sophisticated|technical>",
    "technical_vocabulary": <true|false>,
    "humor_style": "<none|dry|self-deprecating|playful>",
    "warmth_markers": {{
      "examples": ["<actual phrase 1>", "<actual phrase 2>"],
      "pattern": "<abstract description of how warmth is expressed>"
    }},
    "sign_offs": {{
      "examples": ["<actual sign off 1>", "<actual sign off 2>"],
      "pattern": "<abstract description of sign-off style>"
    }}
  }},
  "register_shifts": {{
    "formal_triggers": ["<trigger 1>", "<trigger 2>"],
    "casual_triggers": ["<trigger 1>", "<trigger 2>"],
    "shift_smoothness": "<smooth|abrupt|mixed>"
  }},
  "cognitive_spike": {{
    "coarse_signal": "<one sentence: high-level social/communicative orientation>",
    "mid_signal": "<one sentence: how they structure and organize messages>",
    "fine_signal": "<one sentence: surface-level linguistic and typographic habits>"
  }}
}}

CORPUS:
{corpus}"""

In [ ]:
import anthropic
import json

import os
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=4000,
    system=SYSTEM_PROMPT,
    messages=[
        {
            "role": "user",
            "content": EXTRACTION_PROMPT.format(corpus=corpus)
        }
    ]
)

raw = response.content[0].text.strip()

# Parse JSON
try:
    profile = json.loads(raw)
    print("✅ Profile extracted successfully")
    print(json.dumps(profile, indent=2))
except json.JSONDecodeError as e:
    print(f"❌ JSON parse error: {e}")
    print("Raw output:")
    print(raw)

✅ Profile extracted successfully
{
  "global": {
    "formality_score": 0.35,
    "register": "casual-professional hybrid with context-dependent code-switching",
    "avg_message_length": "short",
    "length_variance": "high",
    "primary_contexts": [
      "professional networking and job search",
      "social coordination and friendship maintenance"
    ],
    "language_mixing": true,
    "overall_tone": "warm, energetic, and earnest with occasional self-deprecating humor",
    "social_orientation": "other-focused",
    "politeness_strategy": "positive politeness"
  },
  "mid_level": {
    "sentence_rhythm": "mixed",
    "avg_sentences_per_message": 3.2,
    "uses_bullet_points": true,
    "uses_numbered_lists": true,
    "paragraph_structure": "short bursts with occasional longer reflective blocks; formal emails use multi-paragraph structure with clear logical progression",
    "question_frequency": "high",
    "rhetorical_questions": true,
    "opener_patterns": {
      "example

In [ ]:
GENERATOR_SYSTEM_PROMPT = """You are a writing style transfer engine.

You are given a detailed psycholinguistic style profile extracted from someone's real writing. Your job is to generate new text that authentically matches their voice, tone, and habits.

Critical rules:
- Use the style profile to internalize patterns, not to copy phrases verbatim
- The examples in the profile are illustrations of underlying habits — generate naturally within those habits
- Separate style from content — match HOW they write, not WHAT they wrote about
- Adapt the register appropriately: if the context is formal, apply their formal register patterns; if casual, apply their casual patterns
- Do not mention or reference the style profile in your output
- Output only the generated text, nothing else
- RESTRAINT IS KEY: real human writing selects from its habits naturally — it does not deploy every stylistic marker at once. Use warmth markers, hedges, and prosodic compensation sparingly and only where they feel organic, not as a checklist
- Avoid stacking multiple warmth signals in the same sentence or paragraph
- Avoid generic AI text markers like emdashes and :
- The output should feel like one natural instance of this person writing, not a compilation of their greatest hits"""

GENERATOR_PROMPT = """Style Profile:
{profile}

Writing Context: {context_type}
Prompt: {prompt}

Generate text that authentically matches this person's voice for the given context and prompt."""

In [ ]:
def generate(prompt, context_type, profile, model="claude-sonnet-4-6"):
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=GENERATOR_SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": GENERATOR_PROMPT.format(
                    profile=json.dumps(profile, indent=2),
                    context_type=context_type,
                    prompt=prompt
                )
            }
        ]
    )
    return response.content[0].text.strip()

In [ ]:
test_cases = [
    {
        "context_type": "formal_email",
        "prompt": "Write a follow-up email to a recruiter at a top AI lab after a first interview, expressing continued interest and attaching your CV"
    },
    {
        "context_type": "slack",
        "prompt": "Message your engineering colleague asking if the new pipeline is ready to test, you've been waiting on it all morning"
    },
    {
        "context_type": "text",
        "prompt": "Text a close friend you haven't seen in a while, suggest grabbing coffee this week"
    }
]

results = []
for test in test_cases:
    output = generate(
        prompt=test["prompt"],
        context_type=test["context_type"],
        profile=profile
    )
    results.append({**test, "output": output})
    print(f"\n{'='*50}")
    print(f"CONTEXT: {test['context_type'].upper()}")
    print(f"PROMPT: {test['prompt']}")
    print(f"\nOUTPUT:\n{output}")
    print('='*50)


CONTEXT: FORMAL_EMAIL
PROMPT: Write a follow-up email to a recruiter at a top AI lab after a first interview, expressing continued interest and attaching your CV

OUTPUT:
Subject: Follow-Up – [Your Name] | [Role Title] Interview

Dear [Recruiter's Name],

I hope you're having a great day! I wanted to follow up on our conversation last week and first just say thank you — it was truly a wonderful opportunity to learn more about the team and the work happening at [Lab Name].

The more I've thought about it since, the more excited I am about this role. I think the intersection of [specific area discussed] with the kind of research [Lab Name] is doing is exactly the direction I want to grow in, and I came away from our conversation feeling genuinely energized.

I've attached my CV here in case it's helpful to have on hand as things move forward. Please let me know if there's anything else I can provide — references, a portfolio, or any additional context about my background.

I truly appre

In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer, util
import numpy as np
import re
import json

# Load the style embedding model — trained specifically for style
# similarity, not semantic similarity
style_model = SentenceTransformer('AnnaWegmann/Style-Embedding')
print("✅ Style embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Style embedding model loaded


In [ ]:
CONTEXT_TO_REGISTER = {
    "formal_email": "EMAIL",
    "slack": "SLACK",
    "text": "WHATSAPP"
}

def parse_corpus_by_register(corpus_text, max_per_register=5000):
    sections = {"WHATSAPP": [], "SLACK": [], "EMAIL": []}
    current = None
    buffer = []

    for line in corpus_text.split('\n'):
        line = line.strip()
        if '[WHATSAPP]' in line:
            current = "WHATSAPP"
        elif '[SLACK]' in line:
            current = "SLACK"
        elif '[EMAIL]' in line:
            current = "EMAIL"
        elif line == '---':
            if current and buffer:
                msg = ' '.join(buffer).strip()
                if len(msg) > 20:
                    sections[current].append(msg)
            buffer = []
        elif line and current:
            buffer.append(line)

    for reg in sections:
        sections[reg] = sections[reg][:max_per_register]
        print(f"  {reg}: {len(sections[reg])} messages")

    return sections

all_messages = parse_corpus_by_register(corpus)
print(f"Total: {sum(len(v) for v in all_messages.values())} messages")

  WHATSAPP: 134 messages
  SLACK: 89 messages
  EMAIL: 36 messages
Total: 259 messages


In [ ]:
emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"
    u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F6FF"
    u"\U0001F1E0-\U0001F1FF"
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

doubling_pattern = re.compile(r'\b(\w+)\s+\1\b', re.IGNORECASE)

def compute_stats(messages):
    if not messages:
        return None

    excl, periods, questions, lengths, emojis, doublings = [], [], [], [], [], []

    for msg in messages:
        words = msg.split()
        wc = max(len(words), 1)
        sents = [s for s in re.split(r'[.!?]+', msg) if s.strip()]
        sc = max(len(sents), 1)

        excl.append(msg.count('!') / wc)
        periods.append(msg.count('.') / wc)
        questions.append(msg.count('?') / sc)
        lengths.append(wc / sc)
        emojis.append(1.0 if emoji_pattern.search(msg) else 0.0)
        doublings.append(1.0 if doubling_pattern.search(msg) else 0.0)

    def stat(vals):
        a = np.array(vals)
        return {"mean": float(a.mean()), "std": float(max(a.std(), 0.001))}

    return {
        "exclamation": stat(excl),
        "period": stat(periods),
        "question": stat(questions),
        "sentence_length": stat(lengths),
        "emoji_rate": float(np.mean(emojis)),
        "doubling_rate": float(np.mean(doublings))
    }

# Compute per register + global
stats_by_register = {}
for reg, msgs in all_messages.items():
    stats_by_register[reg] = compute_stats(msgs)
    if stats_by_register[reg]:
        print(f"{reg} — emoji rate: {stats_by_register[reg]['emoji_rate']:.2f}, "
              f"exclamation mean: {stats_by_register[reg]['exclamation']['mean']:.3f}")

all_msgs_flat = [m for msgs in all_messages.values() for m in msgs]


WHATSAPP — emoji rate: 0.23, exclamation mean: 0.016
SLACK — emoji rate: 0.13, exclamation mean: 0.015
EMAIL — emoji rate: 0.00, exclamation mean: 0.035


In [ ]:
def compute_global_stats_balanced(stats_by_register):
    """
    Average stats across registers with equal weight.
    Prevents WhatsApp dominating global profile.
    """
    keys = ["exclamation", "period", "question", "sentence_length"]
    binary_keys = ["emoji_rate", "doubling_rate"]

    valid = {k: v for k, v in stats_by_register.items()
             if v is not None and k != "GLOBAL"}

    balanced = {}
    for key in keys:
        means = [v[key]["mean"] for v in valid.values()]
        stds = [v[key]["std"] for v in valid.values()]
        balanced[key] = {
            "mean": float(np.mean(means)),
            "std": float(np.mean(stds))
        }

    for key in binary_keys:
        rates = [v[key] for v in valid.values()]
        balanced[key] = float(np.mean(rates))

    return balanced

stats_by_register["GLOBAL"] = compute_global_stats_balanced(stats_by_register)
print(f"GLOBAL — {len(all_msgs_flat)} messages")

GLOBAL — 259 messages


In [ ]:
def sample_stratified(all_messages, n_per_register=15):
    sampled = []
    for reg, msgs in all_messages.items():
        if not msgs:
            continue
        step = max(1, len(msgs) // n_per_register)
        taken = msgs[::step][:n_per_register]
        sampled.extend([(reg, m) for m in taken])
        print(f"  {reg}: {len(taken)} sampled")
    return sampled


def compute_baseline(all_messages, n_per_register=15, n_pairs=50):
    sampled = sample_stratified(all_messages, n_per_register)
    texts = [m for _, m in sampled]
    embeddings = style_model.encode(texts)

    sims = []
    idx = np.random.choice(len(texts), size=(min(n_pairs, len(texts)//2), 2), replace=False)
    for i, j in idx:
        sims.append(float(util.cos_sim(embeddings[i:i+1], embeddings[j:j+1])[0][0]))

    baseline = float(np.mean(sims))
    print(f"Within-corpus baseline similarity: {baseline:.3f}")
    return baseline, sampled

baseline, corpus_sample_pairs = compute_baseline(all_messages)

  WHATSAPP: 15 sampled
  SLACK: 15 sampled
  EMAIL: 15 sampled
Within-corpus baseline similarity: 0.402


In [ ]:
def compute_baselines_by_register(all_messages, n_per_register=15, n_pairs=30):
    """
    Compute within-register style similarity baseline for each register.
    A formal email is normalized against email-to-email similarity,
    not against the global mixed baseline.
    """
    baselines = {}

    for register, msgs in all_messages.items():
        if len(msgs) < 2:
            continue

        # Sample evenly
        step = max(1, len(msgs) // n_per_register)
        sampled = msgs[::step][:n_per_register]

        if len(sampled) < 4:
            print(f"  {register}: too few samples for baseline, skipping")
            continue

        embeddings = style_model.encode(sampled)
        sims = []

        for i in range(len(sampled)):
            for j in range(i+1, len(sampled)):
                sims.append(float(util.cos_sim(
                    embeddings[i:i+1],
                    embeddings[j:j+1]
                )[0][0]))

        baselines[register] = float(np.mean(sims[:n_pairs]))
        print(f"  {register}: baseline = {baselines[register]:.3f} "
              f"({len(sampled)} samples)")

    # Global fallback
    all_flat = [m for msgs in all_messages.values() for m in msgs]
    step = max(1, len(all_flat) // (n_per_register * 3))
    all_sampled = all_flat[::step][:n_per_register * 3]
    all_emb = style_model.encode(all_sampled)
    global_sims = []
    for i in range(len(all_sampled)):
        for j in range(i+1, len(all_sampled)):
            global_sims.append(float(util.cos_sim(
                all_emb[i:i+1], all_emb[j:j+1])[0][0]))
    baselines["GLOBAL"] = float(np.mean(global_sims[:n_pairs]))
    print(f"  GLOBAL: baseline = {baselines['GLOBAL']:.3f}")

    return baselines

baselines_by_register = compute_baselines_by_register(all_messages)

  WHATSAPP: baseline = 0.291 (15 samples)
  SLACK: baseline = 0.482 (15 samples)
  EMAIL: baseline = 0.311 (15 samples)
  GLOBAL: baseline = 0.121


In [ ]:
def score_style_embedding(generated_text, corpus_sample_pairs,
                          baselines_by_register, context_type=None):
    register = CONTEXT_TO_REGISTER.get(context_type, "GLOBAL")
    baseline = baselines_by_register.get(register) or baselines_by_register["GLOBAL"]

    # Compare against register-matched samples only
    texts = [m for reg, m in corpus_sample_pairs if reg == register]
    if len(texts) < 5:
        texts = [m for _, m in corpus_sample_pairs]
        baseline = baselines_by_register["GLOBAL"]

    gen_emb = style_model.encode([generated_text])
    corp_emb = style_model.encode(texts)
    sims = util.cos_sim(gen_emb, corp_emb)[0]
    raw = float(sims.mean())

    normalized = float(np.clip(raw / baseline * 0.85, 0.0, 1.0))

    return {
        "score": normalized,
        "raw": raw,
        "baseline": baseline,
        "register": register
    }

In [ ]:
def gaussian_score(val, mean, std):
    z = (val - mean) / std
    return float(np.exp(-0.5 * z ** 2))

def binary_score(val, rate):
    tendency = rate if rate >= 0.5 else (1 - rate)
    expected = 1.0 if rate >= 0.5 else 0.0
    weight = (tendency - 0.5) * 2
    agreement = 1 if val == expected else -1
    return float(0.5 + 0.5 * weight * agreement)

def score_hedging(text):
    patterns = [
        r'\bjust\b', r'\bmaybe\b', r'\bperhaps\b', r'\bprobably\b',
        r'\bhopefully\b', r'\bi think\b', r'\bi guess\b', r'\bi feel\b',
        r'\bwondering\b', r'\bno rush\b', r'\bno pressure\b',
        r'\bwhenever\b', r'\bif that.s ok\b', r'\bif you have time\b'
    ]
    hits = sum(1 for p in patterns if re.search(p, text.lower()))
    return float(min(hits / 2, 1.0))

def score_programmatic(generated_text, stats_by_register, profile, context_type=None):
    register = CONTEXT_TO_REGISTER.get(context_type, "GLOBAL")
    stats = stats_by_register.get(register) or stats_by_register.get("GLOBAL")

    words = generated_text.split()
    wc = max(len(words), 1)
    sents = [s for s in re.split(r'[.!?]+', generated_text) if s.strip()]
    sc = max(len(sents), 1)

    bd = {}
    bd["exclamation_density"] = gaussian_score(
        generated_text.count('!') / wc, **stats["exclamation"])
    bd["period_usage"] = gaussian_score(
        generated_text.count('.') / wc, **stats["period"])
    bd["question_frequency"] = gaussian_score(
        generated_text.count('?') / sc, **stats["question"])

    raw_len_score = gaussian_score(wc / sc, **stats["sentence_length"])
    bd["sentence_length"] = max(raw_len_score, 0.5)

    has_emoji = 1.0 if emoji_pattern.search(generated_text) else 0.0
    profile_emoji = profile["local"]["emoji_usage"]
    if context_type in ["slack", "text"] and profile_emoji in ["moderate", "frequent"]:
        bd["emoji_present"] = 1.0 if has_emoji else 0.3
    else:
        bd["emoji_present"] = binary_score(has_emoji, stats["emoji_rate"])

    bd["doubling_pattern"] = binary_score(
        1.0 if doubling_pattern.search(generated_text) else 0.0,
        stats["doubling_rate"])
    bd["hedging_present"] = score_hedging(generated_text)

    return {
        "score": float(np.mean(list(bd.values()))),
        "breakdown": bd,
        "register_used": register
    }


LLM_JUDGE_PROMPT = """You are evaluating how well a generated text matches a person's writing style profile.

Score ONLY higher-level style dimensions — register, discourse structure, social orientation, politeness. Do not evaluate surface features like emoji or punctuation.

Style Profile (global + mid level):
{profile_subset}

Generated Text:
{generated_text}

Return ONLY valid JSON, no markdown:
{{
  "register_match": <float 0-1>,
  "information_structure_match": <float 0-1>,
  "social_orientation_match": <float 0-1>,
  "politeness_strategy_match": <float 0-1>,
  "opener_closer_match": <float 0-1>,
  "reasoning": "<one sentence>"
}}"""

def score_llm_judge(generated_text, profile):
    profile_subset = {
        "global": profile["global"],
        "mid_level": profile["mid_level"]
    }
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": LLM_JUDGE_PROMPT.format(
            profile_subset=json.dumps(profile_subset, indent=2),
            generated_text=generated_text
        )}]
    )
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '',
                 response.content[0].text.strip())
    result = json.loads(raw)
    scores = {k: v for k, v in result.items() if k != "reasoning"}
    return {
        "score": float(np.mean(list(scores.values()))),
        "breakdown": scores,
        "reasoning": result.get("reasoning", "")
    }

In [ ]:
def score(generated_text, profile, corpus_sample_pairs,
          baselines_by_register, stats_by_register, context_type=None):

    weights = get_weights(context_type)

    emb = score_style_embedding(generated_text, corpus_sample_pairs,
                                baselines_by_register, context_type)
    prog = score_programmatic(generated_text, stats_by_register,
                              profile, context_type)
    llm = score_llm_judge(generated_text, profile)

    overall = (emb["score"] * weights["embedding"] +
               prog["score"] * weights["programmatic"] +
               llm["score"] * weights["llm"])

    return {
        "overall_score": round(overall, 3),
        "weights_used": weights,
        "style_embedding": emb,
        "programmatic": prog,
        "llm_judge": llm
    }

In [ ]:
for result in results:
    print(f"\n{'='*50}")
    print(f"CONTEXT: {result['context_type'].upper()}")
    print(f"OUTPUT: {result['output'][:100]}...")

    scores = score(result["output"], profile, corpus_sample_pairs,
               baselines_by_register, stats_by_register,
               context_type=result["context_type"])

    print(f"\nOVERALL SCORE: {scores['overall_score']}")
    print(f"  Style Embedding:  {scores['style_embedding']['score']:.3f} "
          f"(raw: {scores['style_embedding']['raw']:.3f}, "
          f"register: {scores['style_embedding']['register']})")
    print(f"  Programmatic:     {scores['programmatic']['score']:.3f} "
          f"(register: {scores['programmatic']['register_used']})")
    print(f"  LLM Judge:        {scores['llm_judge']['score']:.3f}")
    print(f"\n  Programmatic breakdown:")
    for k, v in scores['programmatic']['breakdown'].items():
        print(f"    {k}: {v:.3f}")
    print(f"\n  LLM reasoning: {scores['llm_judge']['reasoning']}")


CONTEXT: FORMAL_EMAIL
OUTPUT: Subject: Follow-Up – [Your Name] | [Role Title] Interview

Dear [Recruiter's Name],

I hope you're h...

OVERALL SCORE: 0.615
  Style Embedding:  0.197 (raw: 0.072, register: EMAIL)
  Programmatic:     0.846 (register: EMAIL)
  LLM Judge:        0.836

  Programmatic breakdown:
    exclamation_density: 0.867
    period_usage: 0.883
    question_frequency: 0.729
    sentence_length: 0.500
    emoji_present: 1.000
    doubling_pattern: 0.944
    hedging_present: 1.000

  LLM reasoning: The generated text aligns well with the profile's formal-email mode — warm opener, context-then-request structure, forward-looking closer ('Have a wonderful day!') — and captures the earnest, other-focused positive politeness, though it lacks the parenthetical asides, doubled expressions, and light self-deprecating humor that distinguish the person's voice even in professional contexts.

CONTEXT: SLACK
OUTPUT: Hey! is the pipeline ready to test? I've been hovering over my ter